# Baseline 2 — Exp10 · Disease-tempered sampling (α = 0.25)

**One-click run: Run All runs ONE controlled Stage 1 experiment (Exp10) and then automatically performs development-only error analysis, class balance, confusion summaries, comparison with Exp09, and writes a leader-ready report.**

- **Only changed training factor vs Exp09:** natural shuffle sampling → disease-frequency tempered weighted sampling (replacement=True, alpha=0.25). Image size 224, B1-aligned preprocessing, weight decay 1e-3, ResNet-50 V2, same seed 42, 4 workers, 9,625 candidates.
- **No automatic Stage 2, official test, 333-candidate restriction, or additional experiments.** Exp11+ are proposed only and will **never** train during Run All.
- **Output:** `Baseline2/outputs/exp10/`. Will **refuse to overwrite** an existing directory.
- **Requirements:** run from `C:\AML\ML_Project\Baseline2` with your previously working CUDA Python kernel; `model_resnet50.py`, `evaluate.py`, `splits_v2/`, and `../DPD_Small_dataset/` must be present.
- **Important:** weighted sampling may repeat a minority image while omitting some others in a given epoch; it does not create new disease diversity. Continue to evaluate on the original unmodified split.

For overnight running, keep the borrowed laptop charging, adequately ventilated on a firm surface, and awake; running this notebook is not guaranteed to complete if it sleeps, disconnects or encounters an error.


## 1. Configuration + safety preflight

The cell below is ready for **Run All** with `RUN_STAGE1 = True`. Only run if you have checked the path, environment and charging/ventilation. The experiment folder is never deleted or overwritten. To repeat after failure, use a NEW experiment ID; don't rerun blindly.


In [2]:
from pathlib import Path
from dataclasses import dataclass
import copy
import gc
import json
import math
import platform
import random
import tempfile
import time

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms as T
from torchvision.transforms import InterpolationMode

from model_resnet50 import MultiTaskResNet50, load_resnet50_checkpoint as load_checkpoint
from evaluate import (DPDDataset, MEAN, STD, evaluate_loader, generalisation_metrics,
                      validate_development_splits, validation_transform,
                      make_candidates, require_candidate_coverage,
                      sha256_file, synthetic_self_test)

PROJECT_DIR = Path.cwd()
SPLIT_DIR = PROJECT_DIR / "splits_v2"
DATASET_ROOT = PROJECT_DIR.parent / "DPD_Small_dataset"   # Configure locally.
IMAGE_ROOT = DATASET_ROOT                # Verified local development image folder; no train/ subfolder.
OUTPUT_ROOT = PROJECT_DIR / "outputs"
RUN_STAGE1 = True   # Run All: exactly ONE new Stage 1 experiment, then development-only audit.

CONFIG = {
    "experiment_id": "exp10", "changed_parameter": "Exp09 matched settings; ONLY disease-frequency tempered sampling (alpha=0.25, replacement=True)",
    "backbone": "resnet50", "pretrained_weights": "IMAGENET1K_V2",
    "crop_classes": 55, "disease_classes": 175, "image_size": 224,
    "batch_size": 16, "max_epochs": 30, "patience": 15, "min_delta": 1e-6,
    "backbone_lr": 1e-5, "head_lr": 1e-3, "crop_dropout": 0.3, "disease_dropout": 0.2,
    "weight_decay": 1e-3, "optimizer": "AdamW", "scheduler": "CosineAnnealingLR",
    "crop_loss_weight": 1.0, "disease_loss_weight": 1.0,
    "seed": 42, "num_workers": 4, "amp": True,
    "sampling_method": "tempered_disease", "sampling_alpha": 0.25,
    "crop_scale": [0.85, 1.0], "flip_probability": 0.5, "vertical_flip_probability": 0.5,
    "rotation_degrees": 10, "brightness": 0.15, "contrast": 0.15,
    "interpolation": "bilinear_resize_nearest_rotation", "mean": list(MEAN), "std": list(STD),
    "candidate_mode": "all_pairs",   # Change ONLY if B1 has a different group-approved candidate rule.
    "candidate_catalog": None,       # For mode='catalog': path to agreed pair-only metadata.
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available(), "device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(json.dumps(CONFIG, indent=2))
# Audit and report automatically follow successful training. No future training is triggered.
RUN_AUTO_AUDIT = True
EXP09_DIR = OUTPUT_ROOT / "b2_resnet50_exp09_b1matched_seed42"
EXP10_DIR = OUTPUT_ROOT / CONFIG["experiment_id"]
assert CONFIG["experiment_id"] == "exp10", "Use this notebook only for Exp10"
assert DEVICE.type == "cuda", "STOP: CUDA GPU is required, do not train overnight on CPU"
assert IMAGE_ROOT.is_dir() and SPLIT_DIR.is_dir(), "Check dataset and split paths"
assert not EXP10_DIR.exists(), f"Already exists: {EXP10_DIR}. Never overwrite saved results."
print("PREFLIGHT: CUDA GPU ready; paths exist; exp10 folder unused. Run All will train one experiment.")


PyTorch: 2.11.0+cu128 torchvision: 0.26.0+cu128
CUDA available: True device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
{
  "experiment_id": "exp10",
  "changed_parameter": "Exp09 matched settings; ONLY disease-frequency tempered sampling (alpha=0.25, replacement=True)",
  "backbone": "resnet50",
  "pretrained_weights": "IMAGENET1K_V2",
  "crop_classes": 55,
  "disease_classes": 175,
  "image_size": 224,
  "batch_size": 16,
  "max_epochs": 30,
  "patience": 15,
  "min_delta": 1e-06,
  "backbone_lr": 1e-05,
  "head_lr": 0.001,
  "crop_dropout": 0.3,
  "disease_dropout": 0.2,
  "weight_decay": 0.001,
  "optimizer": "AdamW",
  "scheduler": "CosineAnnealingLR",
  "crop_loss_weight": 1.0,
  "disease_loss_weight": 1.0,
  "seed": 42,
  "num_workers": 4,
  "amp": true,
  "sampling_method": "tempered_disease",
  "sampling_alpha": 0.25,
  "crop_scale": [
    0.85,
    1.0
  ],
  "flip_probability": 0.5,
  "vertical_flip_probability": 0.5,
  "rotation_degrees": 10,
  "brightness": 0.15,
  "cont

## 2. Verify the exact development metadata
Only `train_fit.csv`, `val_seen.csv`, `val_unseen.csv`, `heldout_compositions.csv`, and `split_manifest.json` are read. Checks include counts, disjoint filenames, canonical ID/name consistency, held-out compositions, concept coverage, and file hashes. No resplitting or relabelling occurs.

Expected: train 6,905 images / 313 compositions; seen validation 695 / 273; unseen validation 400 / 10. Training covers all 55 crops and 175 diseases. Image existence and decoding are checked separately on your laptop.


In [3]:
frames, split_report = validate_development_splits(SPLIT_DIR)
print(json.dumps(split_report, indent=2))
canonical_metadata = pd.concat(frames.values(), ignore_index=True)
class_mappings = {
    task: {str(int(row[0])): str(row[1]) for row in
           canonical_metadata[[f"{task}_id", f"{task}_name"]].drop_duplicates().itertuples(index=False, name=None)}
    for task in ("crop", "disease", "composition")
}
composition_mapping = canonical_metadata[
    ["composition_id", "crop_id", "disease_id", "composition_name"]
].drop_duplicates().sort_values("composition_id").to_dict(orient="records")
print("Original IDs retained; no new class numbering.")


{
  "counts": {
    "train_fit": 6905,
    "val_seen": 695,
    "val_unseen": 400
  },
  "coverage": {
    "train_fit": {
      "crop_id": 55,
      "disease_id": 175,
      "composition_id": 313
    },
    "val_seen": {
      "crop_id": 55,
      "disease_id": 155,
      "composition_id": 273
    },
    "val_unseen": {
      "crop_id": 10,
      "disease_id": 8,
      "composition_id": 10
    }
  },
  "sha256": {
    "train_fit.csv": "fe02d6099c002b10e5a428c2260d5de50f3e677d3b7454216b2a340e55c2a377",
    "val_seen.csv": "ab8d18a13b28fd5a7b9ea531b8ffbdcf4174fd6920337ee9f1a726129390daf5",
    "val_unseen.csv": "b9eae2a57d8331d97636f4e8a8e32bc2e81704d7180fdac22ae923273aadec67",
    "heldout_compositions.csv": "437f57924eef3e5788ed0993caddd969e08f9f86add64791d944425f4bddfe7f",
    "split_manifest.json": "c120d065677e37db6d00880104684c0cd70ccd1c0774ec4aac2eb4b4b0889df8"
  },
  "image_paths_checked": false
}
Original IDs retained; no new class numbering.


## 3. Preprocessing — aligned to Amanda's B1 notebook

Train: `RandomResizedCrop(224, scale=(0.85,1.0))` (bilinear), H/V flip p=.5, rotation +/-10 deg (nearest), brightness/contrast=.15; ImageNet normalization. Validation: `Resize(256)` bilinear, `CenterCrop(224)`, ToTensor, ImageNet normalization. Same all-pair candidate scoring policy (9,625).

For a separate 320-training ablation later, adjust **both** training and evaluation size and use a new experiment ID. Do not call this 224-trained experiment a 320-trained run.


In [4]:
def training_transform(config):
    return T.Compose([
        T.RandomResizedCrop(config["image_size"], scale=tuple(config["crop_scale"]),
                            interpolation=InterpolationMode.BILINEAR),
        T.RandomHorizontalFlip(p=config["flip_probability"]),
        T.RandomVerticalFlip(p=config["vertical_flip_probability"]),
        T.RandomRotation(config["rotation_degrees"], interpolation=InterpolationMode.NEAREST),
        T.ColorJitter(brightness=config["brightness"], contrast=config["contrast"]),
        T.ToTensor(), T.Normalize(MEAN, STD),
    ])

def evaluation_transform(image_size):
    resized = round(image_size * 256 / 224)
    return T.Compose([
        T.Resize(resized, interpolation=InterpolationMode.BILINEAR),
        T.CenterCrop(image_size),
        T.ToTensor(), T.Normalize(MEAN, STD),
    ])


candidates = make_candidates(CONFIG["candidate_mode"], CONFIG["candidate_catalog"])
require_candidate_coverage(candidates, list(frames.values()))
print(f"Candidate mode: {CONFIG['candidate_mode']}; {len(candidates):,} global candidate pairs")
if CONFIG["candidate_mode"] == "all_pairs":
    print("PROVISIONAL: confirm the same candidate policy with Baseline 1 before comparing results.")

Candidate mode: all_pairs; 9,625 global candidate pairs
PROVISIONAL: confirm the same candidate policy with Baseline 1 before comparing results.


## 4. Reproducibility, weighted sampler, DataLoaders and early stopping

**Main causal intervention:** each training example with disease label `d` receives weight `w_i = n_d ** (-0.25)`, where `n_d` is computed **only on 6,905 train-fit images**. The `WeightedRandomSampler` draws `len(train_fit)` samples **with replacement** per epoch; no shuffle argument is given alongside a sampler. Validation loaders remain unweighted and unshuffled. The sampler RNG uses seed 42. The all-pairs candidate policy remains unchanged.

`num_workers=4` came from your benchmark (27.22s for 80 batches), but a loading benchmark is not an end-to-end speed guarantee. The epoch ordering will differ from Exp09 by design.


In [5]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def make_loader(dataset, config, shuffle=False, sampler=None):
    workers = int(config["num_workers"])
    if workers < 0:
        raise ValueError("num_workers must be >= 0")
    generator = torch.Generator().manual_seed(config["seed"])
    if shuffle and sampler is not None:
        raise ValueError("Cannot enable both shuffle and sampler")
    kwargs = dict(batch_size=config["batch_size"], shuffle=shuffle, sampler=sampler,
                  num_workers=workers, pin_memory=DEVICE.type == "cuda",
                  generator=generator, drop_last=False)
    if workers > 0:
        kwargs["prefetch_factor"] = 2
        kwargs["persistent_workers"] = True
    return DataLoader(dataset, **kwargs)


@dataclass
class EarlyStopping:
    patience: int = 5
    min_delta: float = 1e-6
    best: float = -math.inf
    bad_epochs: int = 0

    def update(self, value):
        if not math.isfinite(value):
            raise ValueError("Nonfinite validation harmonic mean")
        improved = value > self.best + self.min_delta
        if improved:
            self.best = value
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return improved, self.bad_epochs >= self.patience


def atomic_checkpoint(payload, path):
    path = Path(path)
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def environment_details():
    return {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__), "numpy": np.__version__,
            "pandas": pd.__version__, "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
            "cuda_runtime": torch.version.cuda}


def make_tempered_disease_sampler(training_frame, alpha=0.25, seed=42):
    """Compute per-image weights exclusively from the existing train-fit labels."""
    if not (0 <= alpha <= 1):
        raise ValueError("alpha must be in [0,1]")
    frequency = training_frame["disease_id"].value_counts()
    if (frequency <= 0).any() or len(training_frame) != 6905:
        raise ValueError("Unexpected train-fit disease frequency / split size")
    per_image_counts = training_frame["disease_id"].map(frequency)
    weights = torch.as_tensor(per_image_counts.to_numpy(dtype=float) ** (-alpha), dtype=torch.double)
    assert len(weights) == len(training_frame) and torch.isfinite(weights).all()
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=weights, num_samples=len(training_frame), replacement=True,
        generator=torch.Generator().manual_seed(seed))
    # Approximate expected draws per label (not a guarantee about realised sampled counts).
    expected = (pd.DataFrame({"disease_id":frequency.index.astype(int),
                 "train_images":frequency.values.astype(int)}).sort_values("disease_id"))
    numerators = expected["train_images"].to_numpy(dtype=float) ** (1-alpha)
    expected["expected_draws_per_epoch"] = len(training_frame) * numerators / numerators.sum()
    return sampler, expected



## 5. Smoke check — model, heads, gradients and checkpoint

Uses **random inputs and no pretrained download**. Runs only a small synthetic check, not real training; the original `evaluate.py` also executes its synthetic metric checks. If you receive a CPU runtime error, check environment before enabling full training.

In [6]:
def run_smoke_checks():
    synthetic_self_test()
    seed_everything(CONFIG["seed"])
    smoke = MultiTaskResNet50(pretrained=False, crop_dropout=CONFIG["crop_dropout"], disease_dropout=CONFIG["disease_dropout"]).to(DEVICE)
    assert smoke.crop_head[1].in_features == smoke.disease_head[1].in_features == 2048
    assert not ({id(p) for p in smoke.crop_head.parameters()} &
                {id(p) for p in smoke.disease_head.parameters()})
    groups = smoke.parameter_groups()
    group_ids = [id(p) for group in groups for p in list(group["params"])]
    assert len(group_ids) == len(set(group_ids)) == len(list(smoke.parameters()))
    # Recreate groups because the generator above was consumed.
    optimizer = torch.optim.AdamW(smoke.parameter_groups(), weight_decay=1e-4)
    # Smoke check uses FP32 for stable gradient assertions; real training still uses CUDA AMP.
    scaler = torch.amp.GradScaler("cuda", enabled=False)
    images = torch.randn(2, 3, 64, 64, device=DEVICE)  # Synthetic check, small to reduce CPU load
    with torch.autocast(device_type=DEVICE.type, enabled=False):
        output = smoke(images)
        assert output["crop_logits"].shape == (2, 55)
        assert output["disease_logits"].shape == (2, 175)
        lc = nn.functional.cross_entropy(output["crop_logits"], torch.tensor([0, 54], device=DEVICE))
        ld = nn.functional.cross_entropy(output["disease_logits"], torch.tensor([0, 174], device=DEVICE))
    shared_parameter = next(smoke.backbone.parameters())
    for loss in (lc, ld):
        gradient = torch.autograd.grad(loss, shared_parameter, retain_graph=True)[0]
        assert torch.isfinite(gradient).all() and gradient.abs().sum() > 0
    loss = lc + ld
    scaler.scale(loss).backward()
    for module in (smoke.backbone, smoke.crop_head, smoke.disease_head):
        grads = [p.grad for p in module.parameters() if p.grad is not None]
        assert grads and all(torch.isfinite(g).all() for g in grads)
    scaler.step(optimizer)
    scaler.update()
    smoke.eval()
    with torch.no_grad():
        before = smoke(images)["crop_logits"].cpu()
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "synthetic.pt"
        atomic_checkpoint({"model_state": smoke.state_dict(), "config": CONFIG}, path)
        restored, _ = load_checkpoint(path, DEVICE)
        with torch.no_grad():
            after = restored(images)["crop_logits"].cpu()
        assert torch.allclose(before, after, atol=1e-6, rtol=1e-5)
    stopper = EarlyStopping()
    assert stopper.update(0) == (True, False)
    for _ in range(4):
        assert stopper.update(0) == (False, False)
    assert stopper.update(0) == (False, True)
    assert stopper.update(0.1) == (True, False)
    assert stopper.update(0.1 + 1e-8) == (False, False)
    del smoke, restored, images, output, loss, lc, ld, optimizer
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print("Architecture, gradients, optimizer, checkpoint, early stopping and metrics passed.")

run_smoke_checks()

Synthetic metric + Exp06 calibration checks passed; no dataset accessed.


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 6. Training and validation functions
The optimiser has separate backbone/head learning rates. Cosine annealing updates once per epoch. CUDA uses autocast and GradScaler; CPU uses float32. Mean epoch losses are weighted by image counts, including the final partial batch. Validation reports both task losses and all six Top-1/Top-5 metrics.

The only selection criterion is **seen-like/unseen-like composition Top-1 harmonic mean**. Each experiment writes to a new folder and refuses to overwrite earlier results.


In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device, amp=True,
                    crop_loss_weight=1.0, disease_loss_weight=1.0):
    model.train()
    totals = np.zeros(3, dtype=float)
    n_images = 0
    criterion = nn.CrossEntropyLoss()
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        crop = batch["crop"].to(device, non_blocking=True)
        disease = batch["disease"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        try:
            with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
                result = model(images)
                crop_loss = criterion(result["crop_logits"], crop)
                disease_loss = criterion(result["disease_logits"], disease)
                total_loss = crop_loss_weight * crop_loss + disease_loss_weight * disease_loss
            if not torch.isfinite(total_loss):
                raise RuntimeError("Nonfinite training loss; stop and inspect the inputs/configuration.")
            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()
        except torch.cuda.OutOfMemoryError as error:
            raise RuntimeError("CUDA out of memory. Restart the kernel, explicitly reduce batch_size to 8, "
                               "record the change, and rerun as a new experiment. No setting was changed automatically.") from error
        count = images.shape[0]
        totals += count * np.array([total_loss.item(), crop_loss.item(), disease_loss.item()])
        n_images += count
    if not n_images:
        raise ValueError("Empty training loader")
    return dict(zip(("total_loss", "crop_loss", "disease_loss"), (totals / n_images).tolist()))


def plot_training_history(history, best_epoch, path):
    table = pd.DataFrame(history)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0, 0].plot(table.epoch, table.train_total_loss, label="Training")
    axes[0, 0].plot(table.epoch, table.seen_total_loss, label="Seen validation")
    axes[0, 0].plot(table.epoch, table.unseen_total_loss, label="Unseen validation")
    axes[0, 0].set(title="Total cross-entropy loss", ylabel="Mean loss")
    axes[0, 1].plot(table.epoch, table.train_crop_loss, label="Crop")
    axes[0, 1].plot(table.epoch, table.train_disease_loss, label="Disease")
    axes[0, 1].set(title="Training task losses", ylabel="Mean loss")
    axes[1, 0].plot(table.epoch, table.seen_composition_top1 * 100, label="Seen-like")
    axes[1, 0].plot(table.epoch, table.unseen_composition_top1 * 100, label="Unseen-like")
    axes[1, 0].set(title="Composition Top-1 validation", ylabel="Accuracy (%)")
    axes[1, 1].plot(table.epoch, table.composition_harmonic_mean * 100, label="Harmonic mean")
    axes[1, 1].set(title="Development selection criterion", ylabel="Accuracy (%)")
    for ax in axes.flat:
        ax.axvline(best_epoch, linestyle="--", color="black", alpha=.5, label=f"Best epoch {best_epoch}")
        ax.set_xlabel("Epoch")
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    return fig


def fit_stage1(config, train_loader, seen_loader, unseen_loader, candidate_pairs,
               metadata_report, mappings, composition_map, output_root=OUTPUT_ROOT):
    seed_everything(config["seed"])
    run_config = copy.deepcopy(config)
    run_config["candidate_catalog"] = (str(config["candidate_catalog"])
                                         if config["candidate_catalog"] is not None else None)
    # A failed pretrained download is an error, never a random-weight fallback.
    model = MultiTaskResNet50(crop_dropout=config["crop_dropout"],
                            disease_dropout=config["disease_dropout"], pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameter_groups(config["backbone_lr"], config["head_lr"]),
                                 weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])
    amp_enabled = config["amp"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    stopper = EarlyStopping(config["patience"], config["min_delta"])
    output_dir = Path(output_root) / config["experiment_id"]
    output_dir.mkdir(parents=True, exist_ok=False)
    environment = environment_details()
    (output_dir / "config.json").write_text(json.dumps({"config": run_config, "environment": environment,
         "development_split": metadata_report, "class_mappings": mappings,
         "composition_mapping": composition_map,
         "candidate_pairs": candidate_pairs.cpu().tolist(),
         "candidate_policy_status": "provisional" if config["candidate_mode"] == "all_pairs" else "user-supplied group catalogue",
         "accuracy_units": "fractions in [0,1]"}, indent=2), encoding="utf-8")
    history = []
    best_epoch = None
    start = time.perf_counter()
    for epoch in range(1, config["max_epochs"] + 1):
        epoch_start = time.perf_counter()
        rates = [group["lr"] for group in optimizer.param_groups]
        training = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, amp_enabled,
                                   config["crop_loss_weight"], config["disease_loss_weight"])
        seen, _ = evaluate_loader(model, seen_loader, DEVICE, candidate_pairs, amp_enabled)
        unseen, _ = evaluate_loader(model, unseen_loader, DEVICE, candidate_pairs, amp_enabled)
        joint = generalisation_metrics(seen, unseen)
        row = {"epoch": epoch, "backbone_lr": rates[0], "head_lr": rates[1],
               **{f"train_{key}": value for key, value in training.items()},
               **{f"seen_{key}": value for key, value in seen.items()},
               **{f"unseen_{key}": value for key, value in unseen.items()},
               "composition_harmonic_mean": joint["composition_harmonic_mean"],
               "seen_unseen_gap": joint["seen_unseen_gap"],
               "epoch_seconds": time.perf_counter() - epoch_start}
        history.append(row)
        pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)
        improved, stop = stopper.update(joint["composition_harmonic_mean"])
        scheduler.step()
        if improved:
            best_epoch = epoch
            best_metrics = {"epoch": epoch, "seen": seen, "unseen": unseen, **joint}
            atomic_checkpoint({"stage": 1, "epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(), "config": run_config,
                "candidate_pairs": candidate_pairs.cpu().tolist(),
                "class_mappings": mappings, "composition_mapping": composition_map,
                "development_split": metadata_report, "environment": environment,
                "best_metrics": best_metrics, "parameter_count": sum(p.numel() for p in model.parameters())},
                output_dir / "best_development.pt")
        print(f"Epoch {epoch:02d} | loss {training['total_loss']:.4f} | "
              f"seen {seen['composition_top1']:.2%} | unseen {unseen['composition_top1']:.2%} | "
              f"HM {joint['composition_harmonic_mean']:.2%}" + (" * best" if improved else ""))
        if stop:
            print(f"Early stopping after {config['patience']} consecutive non-improving epochs.")
            break
    best = torch.load(output_dir / "best_development.pt", map_location=DEVICE, weights_only=True)
    model.load_state_dict(best["model_state"])
    model.eval()
    validation_metrics = {**best["best_metrics"], "epochs_run": len(history),
                          "training_seconds": time.perf_counter() - start,
                          "accuracy_units": "fractions in [0,1]"}
    (output_dir / "validation_metrics.json").write_text(json.dumps(validation_metrics, indent=2), encoding="utf-8")
    log = {"experiment_id": config["experiment_id"], "changed_parameter": config["changed_parameter"],
           "best_epoch": best_epoch, **generalisation_metrics(best["best_metrics"]["seen"], best["best_metrics"]["unseen"]),
           "candidate_mode": config["candidate_mode"], "checkpoint_path": str(output_dir / "best_development.pt"),
           "configuration": json.dumps(run_config, sort_keys=True)}
    pd.DataFrame([log]).to_csv(output_dir / "experiment_summary.csv", index=False)
    figure = plot_training_history(history, best_epoch, output_dir / "training_curves.png")
    plt.show()
    return model, history, validation_metrics, output_dir

## 7. Exp10 Stage 1 — **automatically runs**

Trains once on existing `train_fit` only, validates against `val_seen`/`val_unseen`, selects checkpoint by **development composition HM**, and saves `outputs/exp10/best_development.pt` (best epoch). Uses unmodified cosine scheduler and patience=15. Explicitly compares the class distributions under natural vs tempered sampling, but no test labels are used. If interrupted, inspect partial `training_history.csv` and checkpoint before deciding on a new run ID.


In [ ]:
if RUN_STAGE1:
    if DEVICE.type != "cuda":
        raise RuntimeError("CUDA GPU is required for this run. Stop rather than training for hours on CPU.")
    if (OUTPUT_ROOT / CONFIG["experiment_id"]).exists():
        raise FileExistsError("Experiment folder already exists. Review it; use a new experiment ID for an approved rerun.")
    frames, verified_report = validate_development_splits(SPLIT_DIR)
    candidates = make_candidates(CONFIG["candidate_mode"], CONFIG["candidate_catalog"])
    require_candidate_coverage(candidates, list(frames.values()))
    datasets = {
        name: DPDDataset(frame, IMAGE_ROOT,
                         training_transform(CONFIG) if name == "train_fit" else evaluation_transform(CONFIG["image_size"]))
        for name, frame in frames.items()
    }
    # Check decoding for CSV-listed training/development images only.
    for dataset in datasets.values():
        for path in dataset.paths:
            with Image.open(path) as image:
                image.convert("RGB").load()
    verified_report["image_paths_checked"] = True
    verified_report["image_decoding_checked"] = True
    sampler, exposure_table = make_tempered_disease_sampler(
        frames["train_fit"], alpha=CONFIG["sampling_alpha"], seed=CONFIG["seed"])
    print("Tempered sampling enabled: alpha=", CONFIG["sampling_alpha"],
          "samples/epoch=", len(frames["train_fit"]))
    loaders = {name: make_loader(dataset, CONFIG, sampler=sampler if name == "train_fit" else None)
               for name, dataset in datasets.items()}
    seed_everything(CONFIG["seed"])
    example = next(iter(loaders["val_seen"]))
    assert example["image"].shape[1:] == (3, CONFIG["image_size"], CONFIG["image_size"])
    model, history, metrics, experiment_dir = fit_stage1(
        CONFIG, loaders["train_fit"], loaders["val_seen"], loaders["val_unseen"],
        candidates, verified_report, class_mappings, composition_mapping)
    print("Best development epoch:", metrics["epoch"])
    exposure_table.to_csv(experiment_dir / "expected_disease_sampling_exposure.csv", index=False)
    print("Saved actual results to:", experiment_dir)
    # Close persistent 4-worker loaders and free the training model before post-training audit.
    # This reduces unnecessary CPU/GPU memory pressure on a 16 GB borrowed laptop.
    del loaders, datasets, example, sampler, model
    gc.collect()
    plt.close('all')
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print("Stage 1 complete; training loaders closed. Beginning development-only audit.")
else:
    raise RuntimeError("Stage 1 is disabled; run all is configured only for a deliberate Exp10 training.")

## 8. Automatic Exp10 error analysis — **runs after successful training**

Uses only the saved **best development checkpoint**, checks split hashes, exports seen/unseen prediction CSVs and per-split head/error metrics. This does **not** perform official test evaluation.


In [ ]:
if RUN_AUTO_AUDIT:
    ckpt = EXP10_DIR / "best_development.pt"
    if not ckpt.is_file():
        raise FileNotFoundError("Exp10 checkpoint missing; cannot audit")
    diagnostic_model, saved = load_checkpoint(ckpt, DEVICE)
    assert saved["stage"] == 1 and saved["config"]["experiment_id"] == "exp10"
    assert saved["development_split"]["sha256"] == split_report["sha256"], "Split mismatch"
    saved_candidates = torch.as_tensor(saved["candidate_pairs"], dtype=torch.long)
    train_pair_set = set(map(tuple, frames["train_fit"][["crop_id", "disease_id"]].to_numpy().tolist()))
    predictions, audit_summary = {}, []
    for split in ("val_seen", "val_unseen"):
        ds = DPDDataset(frames[split], IMAGE_ROOT, evaluation_transform(CONFIG["image_size"]))
        dl = make_loader(ds, saved["config"], shuffle=False)
        metrics_split, records = evaluate_loader(diagnostic_model, dl, DEVICE, saved_candidates,
                                                   amp=saved["config"]["amp"], collect_predictions=True)
        tab = pd.DataFrame(records)
        assert len(tab) == len(frames[split]) and tab.filename.is_unique
        tab["crop_correct"] = tab.crop_id == tab.predicted_crop_id
        tab["disease_correct"] = tab.disease_id == tab.predicted_disease_id
        tab["crop_only_wrong"] = ~tab.crop_correct & tab.disease_correct
        tab["disease_only_wrong"] = tab.crop_correct & ~tab.disease_correct
        tab["both_wrong"] = ~tab.crop_correct & ~tab.disease_correct
        tab["predicted_pair_seen_in_training"] = [
            (int(c), int(d)) in train_pair_set
            for c,d in zip(tab.predicted_pair_crop_id, tab.predicted_pair_disease_id)]
        if int((tab.crop_correct & tab.disease_correct).sum()) != round(metrics_split["composition_top1"] * len(tab)):
            print("Warning: independent head argmax and composition ranking differ (possible score ties).")
        predictions[split] = tab
        tab.to_csv(EXP10_DIR / f"{split}_error_predictions.csv", index=False)
        audit_summary.append({"split":split, "n":len(tab),
            "crop_top1":metrics_split["crop_top1"],
            "disease_top1":metrics_split["disease_top1"],
            "composition_top1":metrics_split["composition_top1"],
            "composition_top5":metrics_split["composition_top5"],
            "disease_only_wrong_count":int(tab.disease_only_wrong.sum()),
            "crop_only_wrong_count":int(tab.crop_only_wrong.sum()),
            "both_wrong_count":int(tab.both_wrong.sum()),
            "predicted_pair_seen_in_training_rate":float(tab.predicted_pair_seen_in_training.mean())})
    audit_table = pd.DataFrame(audit_summary)
    audit_table.to_csv(EXP10_DIR / "task_error_breakdown.csv",index=False)
    display(audit_table)
    print("Saved Exp10 predictions + error breakdown")
else:
    print("Auto audit disabled. No evaluation performed.")


## 9. Automatic class counts + per-class accuracy (weakest and strongest)

Creates one CSV for each of crop/disease/composition training counts and per-class CSVs for both validation splits. The per-class accuracy tables use the true validation support `n_validation` and restrict displayed top/bottom to `n>=5`. Minority disease performance must be interpreted alongside validation support.


In [ ]:
if RUN_AUTO_AUDIT:
    class_counts = {}
    for task in ("crop", "disease", "composition"):
        idcol, namecol = f"{task}_id", f"{task}_name"
        table = (frames["train_fit"].groupby([idcol, namecol], dropna=False)
                 .size().rename("train_images").reset_index()
                 .sort_values(["train_images",namecol], ascending=[True,True]))
        for split in ("val_seen", "val_unseen"):
            counts = frames[split].groupby(idcol).size().rename(f"{split}_images")
            table = table.join(counts,on=idcol)
        table[["val_seen_images","val_unseen_images"]] = table[["val_seen_images","val_unseen_images"]].fillna(0).astype(int)
        table.to_csv(EXP10_DIR / f"{task}_class_counts.csv",index=False)
        class_counts[task] = table
        print(f"{task}: {len(table)} labels | min {int(table.train_images.min())} | median {table.train_images.median():g} | max {int(table.train_images.max())}")

    MIN_SUPPORT = 5
    per_class = {}
    for split, pred in predictions.items():
        for task in ("crop", "disease"):
            idcol, namecol, predcol = f"{task}_id", f"{task}_name", f"predicted_{task}_id"
            rows = pred.assign(_correct=(pred[idcol]==pred[predcol]).astype(int))
            score = rows.groupby(idcol).agg(n_validation=(idcol,"size"), n_correct=("_correct","sum")).reset_index()
            score["top1_accuracy"] = score.n_correct / score.n_validation
            names = class_counts[task][[idcol,namecol,"train_images"]]
            score=score.merge(names,on=idcol,how="left",validate="one_to_one")
            score["n_wrong"] = score.n_validation - score.n_correct
            errs = rows.loc[rows._correct==0]
            frequent_wrong = (errs.groupby([idcol,predcol]).size().rename("mistake_count")
               .reset_index().sort_values("mistake_count",ascending=False).drop_duplicates(idcol))
            freq_names = dict(zip(names[idcol],names[namecol]))
            frequent_wrong["most_common_wrong_prediction"] = frequent_wrong[predcol].map(freq_names)
            score=score.merge(frequent_wrong[[idcol,"most_common_wrong_prediction","mistake_count"]],
                              on=idcol,how="left")
            score=score.sort_values(["top1_accuracy","n_validation"],ascending=[True,False])
            per_class[(split,task)]=score
            score.to_csv(EXP10_DIR / f"{split}_{task}_per_class.csv",index=False)
            eligible=score.loc[score.n_validation>=MIN_SUPPORT]
            print(f"\n{split} {task}: {len(eligible)} classes with n >= {MIN_SUPPORT}")
            display(eligible[[namecol,"train_images","n_validation","top1_accuracy","most_common_wrong_prediction"]].head(6))
            display(eligible.sort_values(["top1_accuracy","n_validation"],ascending=[False,False])
                    [[namecol,"train_images","n_validation","top1_accuracy","most_common_wrong_prediction"]].head(6))
else:
    print("Audit disabled")


## 10. Automatic confusion matrix CSVs + readable weak/strong visualisations

Saves full 55×55 and 175×175 head confusion counts (all canonical classes, including zero-support rows), plus condensed `weakest` / `strongest` rows. Correct counts are counted once, and globally common wrong labels exclude per-row correct predictions. The result is an inspectable **head-level error summary**; these are not exact composition confusion matrices.


In [ ]:
if RUN_AUTO_AUDIT:
    for split, pred in predictions.items():
        for task in ("crop","disease"):
            idcol, namecol, predcol = f"{task}_id", f"{task}_name", f"predicted_{task}_id"
            freq = class_counts[task]
            id_to_name=dict(zip(freq[idcol],freq[namecol]))
            all_ids=sorted(freq[idcol].unique().tolist())
            full = pd.crosstab(pred[idcol],pred[predcol]).reindex(index=all_ids,columns=all_ids,fill_value=0)
            full.index=[f"{i}: {id_to_name[i]}" for i in all_ids]
            full.columns=[f"{i}: {id_to_name[i]}" for i in all_ids]
            full.to_csv(EXP10_DIR / f"{split}_{task}_full_confusion.csv")
            qualified=per_class[(split,task)].query("n_validation >= @MIN_SUPPORT")
            selections={"weakest":qualified.head(6),
                        "strongest":qualified.sort_values(["top1_accuracy","n_validation"],ascending=[False,False]).head(6)}
            for label,selected in selections.items():
                chosen=selected[idcol].tolist()
                small=pred.loc[pred[idcol].isin(chosen),[idcol,predcol]].copy()
                common=(small.loc[small[idcol]!=small[predcol],predcol]
                        .value_counts().head(5).index.tolist())
                columns=["Correct"]+[f"{int(p)}: {id_to_name.get(int(p),p)}" for p in common]+["Other wrong"]
                counts, labels=[],[]
                for target in chosen:
                    part=small.loc[small[idcol]==target]
                    vals=[int((part[predcol]==target).sum())]
                    vals += [int(((part[predcol]==p)&(part[predcol]!=target)).sum()) for p in common]
                    vals.append(int(((part[predcol]!=target)&(~part[predcol].isin(common))).sum()))
                    assert sum(vals)==len(part), "Confusion row counts do not match support"
                    counts.append(vals)
                    labels.append(f"{id_to_name[target]} (n={len(part)})")
                matrix=pd.DataFrame(counts,index=labels,columns=columns)
                matrix.to_csv(EXP10_DIR / f"{split}_{task}_{label}_confusion_summary.csv")
                fig,ax=plt.subplots(figsize=(max(8,2+len(columns)*1.2),max(4, len(chosen)*0.6+1)))
                im=ax.imshow(matrix.to_numpy(), aspect="auto")
                ax.set_xticks(np.arange(len(columns)),labels=columns,rotation=45,ha="right",fontsize=8)
                ax.set_yticks(np.arange(len(labels)),labels=labels,fontsize=9)
                for (i,j),value in np.ndenumerate(matrix.to_numpy()):
                    ax.text(j,i,str(value),ha="center",va="center",fontsize=8)
                ax.set_title(f"Exp10 {split} {task} — {label} classes (counts)")
                fig.colorbar(im,ax=ax,label="Images")
                fig.tight_layout()
                fig.savefig(EXP10_DIR / f"{split}_{task}_{label}_confusion_summary.png",dpi=135,bbox_inches="tight")
                plt.close(fig)
    print("Saved all eight weak/strong confusion summary charts, CSVs and four full confusion matrices")
else:
    print("Audit disabled")


## 11. Automatic comparison vs Exp09 + leader-ready report

The comparison uses each experiment's **best development composition-HM checkpoint**, not its final epoch. Reports the number of correct unseen compositions (out of 400), crop vs disease head accuracies, seen-pair preference, per-class diagnostic findings and the training change. Exp09 source is read-only. Missing Exp09 analysis files are noted rather than fabricated.


In [ ]:
if RUN_AUTO_AUDIT:
    m10=json.loads((EXP10_DIR / "validation_metrics.json").read_text(encoding="utf-8"))
    p09=EXP09_DIR / "validation_metrics.json"
    if not p09.is_file():
        print("Exp09 validation_metrics.json not found; summary will still cover Exp10.")
        m09=None
    else:
        m09=json.loads(p09.read_text(encoding="utf-8"))
    comparisons=[]
    for name,metric in [("Exp09 natural shuffle",m09),("Exp10 tempered alpha=0.25",m10)]:
        if metric is None:
            continue
        comparisons.append({"experiment":name, "best_epoch":int(metric["epoch"]),
             "seen_composition":metric["seen"]["composition_top1"],
             "unseen_composition":metric["unseen"]["composition_top1"],
             "composition_HM":metric["composition_harmonic_mean"],
             "unseen_crop":metric["unseen"]["crop_top1"],
             "unseen_disease":metric["unseen"]["disease_top1"],
             "unseen_correct_of_400":round(metric["unseen"]["composition_top1"] * 400)})
    cmp=pd.DataFrame(comparisons)
    cmp.to_csv(EXP10_DIR / "exp09_vs_exp10_development.csv", index=False)
    display(cmp)
    e10_unseen=audit_table.set_index("split").loc["val_unseen"]
    exp09_breakdown_file=EXP09_DIR / "task_error_breakdown.csv"
    exp09_audit=None
    if exp09_breakdown_file.is_file():
        exp09_audit=pd.read_csv(exp09_breakdown_file).set_index("split")
    p10=per_class[("val_unseen","disease")].sort_values(["top1_accuracy","n_validation"])
    p10_weak=p10.loc[p10.n_validation>=5].head(8)
    # Optional paired per-class changes if the user's previously completed Exp09 audit exists.
    old_class_file = OUTPUT_ROOT / "exp09_audit" / "val_unseen_disease_per_class.csv"
    change_table = None
    if old_class_file.is_file():
        old_c = pd.read_csv(old_class_file)[["disease_id","n_validation","top1_accuracy"]].rename(
            columns={"n_validation":"n_validation_exp09","top1_accuracy":"exp09_accuracy"})
        new_c = per_class[("val_unseen","disease")][["disease_id","disease_name","train_images","n_validation","top1_accuracy"]].rename(
            columns={"n_validation":"n_validation_exp10","top1_accuracy":"exp10_accuracy"})
        change_table = new_c.merge(old_c,on="disease_id",how="inner",validate="one_to_one")
        assert (change_table.n_validation_exp10 == change_table.n_validation_exp09).all(), "Validation label counts changed"
        change_table["accuracy_delta_pp"] = (change_table.exp10_accuracy-change_table.exp09_accuracy)*100
        change_table.to_csv(EXP10_DIR / "exp09_vs_exp10_unseen_disease_per_class.csv",index=False)
    def markdown_table(data, columns, percent_columns=()):
        head="| " + " | ".join(columns) + " |"
        divider="| " + " | ".join(["---"]*len(columns)) + " |"
        lines=[head,divider]
        for _,row in data.iterrows():
            cells=[]
            for col in columns:
                val=row[col]
                if col in percent_columns and pd.notna(val):
                    cells.append(f"{float(val):.2%}")
                elif isinstance(val,(float,np.floating)):
                    cells.append(f"{float(val):.2f}")
                else:
                    cells.append(str(val))
            lines.append("| " + " | ".join(cells) + " |")
        return "\n".join(lines)
    r=[]
    r.append("# Baseline 2 Exp10 — Results for team leader (development only)")
    r.append("")
    r.append("## Objective and controlled change")
    r.append("- Same multi-task ResNet-50 ImageNet-1K V2, two heads, 224 image size, augmentation, 9,625 candidates, seed 42, weight decay 1e-3, batch 16, 4 workers, 6,905/695/400 split as Exp09.")
    r.append("- Exp09 uses shuffled sampling; Exp10 changes ONLY the train sampler to disease-frequency weights n_d^(-0.25), with replacement and 6,905 draws per epoch. Exp10 training examples may repeat; validation unchanged.")
    r.append("- Stage 1 checkpoint selected by development composition HM; official test not accessed; Stage 2 not run.")
    r.append("")
    r.append("## Main metrics — best development checkpoints")
    r.append(markdown_table(cmp, ["experiment","best_epoch","seen_composition","unseen_composition","composition_HM","unseen_crop","unseen_disease","unseen_correct_of_400"],
                            percent_columns=("seen_composition","unseen_composition","composition_HM","unseen_crop","unseen_disease")))
    r.append("")
    r.append("## Exp10 error diagnosis")
    r.append(f"- Unseen crop Top-1: {e10_unseen['crop_top1']:.2%}; unseen disease Top-1: {e10_unseen['disease_top1']:.2%}; composition Top-1: {e10_unseen['composition_top1']:.2%}.")
    r.append(f"- Disease-only wrong: {int(e10_unseen['disease_only_wrong_count'])} / 400; crop-only wrong: {int(e10_unseen['crop_only_wrong_count'])} / 400; both wrong: {int(e10_unseen['both_wrong_count'])} / 400.")
    r.append(f"- Predicted training-seen pairs on held-out combinations: {e10_unseen['predicted_pair_seen_in_training_rate']:.2%}.")
    if exp09_audit is not None:
        e09=exp09_audit.loc["val_unseen"]
        r.append(f"- Exp09 comparison: unseen disease {e09['disease_top1']:.2%}, disease-only errors {int(e09['disease_only_wrong_count'])}, predicted seen-pair rate {e09['predicted_pair_seen_in_training_rate']:.2%}.")
    else:
        r.append("- Exp09 task_error_breakdown.csv unavailable; compare diagnostic error counts manually using saved Exp09 Section 8 output.")
    r.append("")
    r.append("## Weakest unseen disease classes (support >=5)")
    r.append(markdown_table(p10_weak, ["disease_name","train_images","n_validation","n_correct","top1_accuracy","most_common_wrong_prediction"],
                             percent_columns=("top1_accuracy",)))
    if change_table is not None:
        r.append("")
        r.append("## Exp09 to Exp10 per-disease changes (same validation images)")
        r.append("**Largest development improvements, only classes with n>=5:**")
        r.append(markdown_table(change_table.loc[change_table.n_validation_exp10>=5].nlargest(6,"accuracy_delta_pp"),
            ["disease_name","train_images","n_validation_exp10","exp09_accuracy","exp10_accuracy","accuracy_delta_pp"],
            percent_columns=("exp09_accuracy","exp10_accuracy")))
        r.append("**Largest development declines, only classes with n>=5:**")
        r.append(markdown_table(change_table.loc[change_table.n_validation_exp10>=5].nsmallest(6,"accuracy_delta_pp"),
            ["disease_name","train_images","n_validation_exp10","exp09_accuracy","exp10_accuracy","accuracy_delta_pp"],
            percent_columns=("exp09_accuracy","exp10_accuracy")))
    r.append("")
    r.append("## Class imbalance evidence from TRAIN-FIT (not test)")
    for task in ("crop","disease","composition"):
        c=class_counts[task]
        r.append(f"- {task}: {len(c)} classes/pairs; min {int(c.train_images.min())}, median {c.train_images.median():g}, max {int(c.train_images.max())} images.")
    r.append("")
    r.append("## Interpretation / limits")
    r.append("- Higher class balancing may help some rare diseases but can hurt crop accuracy and may oversample single examples; this experiment alone does not prove a causal explanation for unseen-pair errors.")
    r.append("- Results are on one development split and one seed. Compare paired per-class errors and repeat with different fixed seeds before claiming robust improvements.")
    r.append("- Candidate restriction to 333 was NOT tested; requires an approved independently specified catalogue. Changing candidate count is a separate inference-only protocol, not training.")
    r.append("")
    r.append("## Saved evidence")
    r.append("- training_history.csv, training_curves.png, best_development.pt, validation_metrics.json, task_error_breakdown.csv, class counts, per-class accuracy, confusion matrices and summary CSV are in outputs/exp10/.")
    text="\n".join(r)+"\n"
    (EXP10_DIR / "leader_summary.md").write_text(text, encoding="utf-8")
    print("LEADER SUMMARY written to:", EXP10_DIR / "leader_summary.md")
    print("SUCCESS: Exp10 training + development audit completed; no official test or Stage 2 executed.")
else:
    print("Audit disabled")


## 12. Later experiments — planning only, **NO automatic training**

The following changes are separate hypotheses, **not all variants of Exp10**. Select one next after reviewing Exp10 and Exp09 results; do not combine changes or choose a seed based on best observed validation score. Stage 2/official test and restricted 333-candidate evaluation remain excluded.

| Future study | Isolate variable | Expected artefacts | Caveat |
|---|---|---|---|
| Exp11: multi-seed | same final configuration, seeds 123 and 2026 (and reuse seed 42 only if configuration identical) | mean ± SD of composition HM and unseen disease | each seed retrains independently; this notebook does **not** start them |
| Exp12: loss ablation | crop loss 1.0 vs disease loss 1.5 on Exp09 reference | disease/crop trade-off, HM, paired errors | compare only with same preprocessing and sampling |
| Exp13: 320 training | train resolution 320 (vs 224) | seen/unseen accuracy and GPU cost | 320 *inference only* did not increase HM for Exp09 |
| Post-hoc seen-pair calibration | use fixed checkpoint and train-fit pair list; tune gamma using appropriate development protocol | seen/unseen trade-off curve | no retraining, no official-test-based tuning |
| 333 candidates | approved external 333 catalogue (if evaluation policy permits) | all 9,625 vs fixed 333 scoring | no retraining; do not construct catalogue from held-out/test answers |

**All these are unexecuted**. Results must not be claimed before they are actually measured.
